# EXEMPLO 2: RRF vs Normalization

**Objetivo**: Comparar duas estratégias de rank fusion para busca híbrida

Este notebook demonstra:
- Pipeline RRF (Reciprocal Rank Fusion)
- Pipeline com Min-Max Normalization
- Comparação lado a lado
- Tabela de quando usar cada técnica

## Setup

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'opensearch-py', 'pandas', 'matplotlib'])

from opensearchpy import OpenSearch
from typing import List, Dict
import pandas as pd
import matplotlib.pyplot as plt
import hashlib

# Conexão
client = OpenSearch(
    hosts=[{'host': 'localhost', 'port': 9200}],
    http_auth=('admin', 'admin'),
    use_ssl=False, verify_certs=False
)

print(f"✓ Conectado ao OpenSearch {client.info()['version']['number']}")

## Criar Índice e Dados

In [ ]:
index_name = "indice_comparacao_fusion"

# Limpar
try:
    client.indices.delete(index=index_name)
except:
    pass

# Criar índice
mapping = {
    "settings": {"number_of_shards": 1, "index": {"knn": True}},
    "mappings": {
        "properties": {
            "titulo": {"type": "text"},
            "conteudo": {"type": "text"},
            "embedding": {"type": "knn_vector", "dimension": 384, "method": {"name": "hnsw", "space_type": "cosinesimil", "engine": "lucene"}}
        }
    }
}

client.indices.create(index=index_name, body=mapping)

# Função de embedding
def simple_embedding(text: str, dim: int = 384) -> List[float]:
    h = hashlib.md5(text.encode()).hexdigest()
    return [(int(h[i % len(h)], 16) % 100) / 100.0 for i in range(dim)]

# Dados
docs = [
    {"id": "1", "titulo": "Acesso à Informação Pública", "conteudo": "Lei de Acesso à Informação garante direito irrestrito a informações públicas"},
    {"id": "2", "titulo": "Proteção de Dados Pessoais", "conteudo": "LGPD protege dados pessoais e privacidade dos cidadãos brasileiros"},
    {"id": "3", "titulo": "Direitos Fundamentais", "conteudo": "Constituição Federal garante vida, liberdade, igualdade, segurança e propriedade"},
    {"id": "4", "titulo": "Responsabilidade Civil do Estado", "conteudo": "Estado responde por danos causados por seus agentes públicos"},
    {"id": "5", "titulo": "Segurança Pública", "conteudo": "Segurança pública é direito e responsabilidade de todos, respeitando direitos fundamentais"},
]

# Indexar
for doc in docs:
    doc["embedding"] = simple_embedding(doc["conteudo"])
    body = {"titulo": doc["titulo"], "conteudo": doc["conteudo"], "embedding": doc["embedding"]}
    client.index(index=index_name, id=doc["id"], body=body)

client.indices.refresh(index=index_name)
print(f"✓ {len(docs)} documentos indexados em '{index_name}'")

## Criar Pipelines: RRF vs Normalization

In [ ]:
# Pipeline 1: RRF (Reciprocal Rank Fusion)
# Combina ranks inversos: score = 1/(k + rank)
rrf_pipeline = {
    "description": "RRF - Reciprocal Rank Fusion",
    "processors": [{
        "rrf": {
            "weights": [0.5, 0.5]  # Pesos iguais para BM25 e KNN
        }
    }]
}

# Pipeline 2: Min-Max Normalization
# Normaliza scores para [0,1] antes de combinar
normalization_pipeline = {
    "description": "Min-Max Normalization",
    "processors": [{
        "normalization-processor": {
            "normalization": "min_max",
            "combination": {
                "method": "arithmetic_mean",
                "weights": [0.5, 0.5]
            }
        }
    }]
}

# Deletar pipelines anteriores
for name in ['rrf_pipeline_exemplo2', 'normalization_pipeline_exemplo2']:
    try:
        client.transport.perform_request('DELETE', f'/_search/pipeline/{name}')
    except:
        pass

# Criar pipelines
try:
    client.transport.perform_request('PUT', '/_search/pipeline/rrf_pipeline_exemplo2', body=rrf_pipeline)
    print("✓ Pipeline RRF criado")
except Exception as e:
    print(f"⚠ Pipeline RRF: {str(e)[:100]}")

try:
    client.transport.perform_request('PUT', '/_search/pipeline/normalization_pipeline_exemplo2', body=normalization_pipeline)
    print("✓ Pipeline Normalization criado")
except Exception as e:
    print(f"⚠ Pipeline Normalization: {str(e)[:100]}")

## Query Híbrida - Executar em Ambos os Pipelines

In [ ]:
def execute_hybrid_query(client, index: str, query_text: str, pipeline_name: str, size: int = 5):
    """
    Executa query híbrida com pipeline especificado.
    """
    query_emb = simple_embedding(query_text)
    
    search_body = {
        "size": size,
        "query": {
            "bool": {
                "should": [
                    # BM25: busca por palavras-chave
                    {"multi_match": {
                        "query": query_text,
                        "fields": ["titulo^2", "conteudo"],
                        "fuzziness": "AUTO"
                    }},
                    # KNN: busca semântica
                    {"knn": {
                        "embedding": {
                            "vector": query_emb,
                            "k": size
                        }
                    }}
                ]
            }
        },
        "ext": {"search_pipeline": pipeline_name}
    }
    
    try:
        response = client.search(index=index, body=search_body)
        results = []
        for i, hit in enumerate(response['hits']['hits'], 1):
            results.append({
                'rank': i,
                'id': hit['_id'],
                'score': hit['_score'],
                'titulo': hit['_source']['titulo']
            })
        return results
    except Exception as e:
        print(f"Erro na query {pipeline_name}: {str(e)[:100]}")
        return []

# Executar query em ambos os pipelines
test_query = "informação pública e segurança"

results_rrf = execute_hybrid_query(client, index_name, test_query, 'rrf_pipeline_exemplo2', size=5)
results_norm = execute_hybrid_query(client, index_name, test_query, 'normalization_pipeline_exemplo2', size=5)

print(f"\nQuery: '{test_query}'")
print(f"\nResultados RRF: {len(results_rrf)} documentos")
print(f"Resultados Normalization: {len(results_norm)} documentos")

## Tabela Comparativa de Scores e Rankings

In [ ]:
# Criar DataFrame comparativo
df_rrf = pd.DataFrame(results_rrf) if results_rrf else pd.DataFrame()
df_norm = pd.DataFrame(results_norm) if results_norm else pd.DataFrame()

# Tabela lado a lado
print("\n" + "="*80)
print("RRF - RECIPROCAL RANK FUSION")
print("="*80)
if len(df_rrf) > 0:
    df_display_rrf = df_rrf[['rank', 'id', 'titulo', 'score']].copy()
    print(df_display_rrf.to_string(index=False))
else:
    print("(Nenhum resultado)")

print("\n" + "="*80)
print("MIN-MAX NORMALIZATION")
print("="*80)
if len(df_norm) > 0:
    df_display_norm = df_norm[['rank', 'id', 'titulo', 'score']].copy()
    print(df_display_norm.to_string(index=False))
else:
    print("(Nenhum resultado)")

# Mostrar diferenças
print("\n" + "="*80)
print("ANÁLISE DE DIFERENÇAS")
print("="*80)
if len(df_rrf) > 0 and len(df_norm) > 0:
    # Comparar ordem de ranking
    print("\nDocumento | RRF Rank | Norm Rank | Diferença")
    print("-" * 50)
    for doc_id in df_rrf['id'].unique():
        rrf_rank = df_rrf[df_rrf['id'] == doc_id]['rank'].values[0] if doc_id in df_rrf['id'].values else '-'
        norm_rank = df_norm[df_norm['id'] == doc_id]['rank'].values[0] if doc_id in df_norm['id'].values else '-'
        if isinstance(rrf_rank, int) and isinstance(norm_rank, int):
            diff = abs(rrf_rank - norm_rank)
            print(f"Doc {doc_id:1s} | {rrf_rank:8d} | {norm_rank:9d} | {diff:9d}")
else:
    print("(Insuficientes dados para comparação)")

## Gráfico Comparativo de Scores

In [ ]:
# Preparar dados para gráfico
if len(df_rrf) > 0 and len(df_norm) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Gráfico 1: Scores RRF vs Normalization
    ax = axes[0]
    x = range(len(df_rrf))
    ax.plot(x, df_rrf['score'].values, marker='o', label='RRF', linewidth=2, markersize=8)
    ax.plot(x, df_norm['score'].values, marker='s', label='Normalization', linewidth=2, markersize=8)
    ax.set_xlabel('Rank (posição)')
    ax.set_ylabel('Score')
    ax.set_title('Scores por Estratégia de Fusion')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xticks(x)
    
    # Gráfico 2: Documentos ranking
    ax = axes[1]
    docs = [f"Doc {id}" for id in df_rrf['id'].values]
    x = range(len(docs))
    width = 0.35
    ax.bar([i - width/2 for i in x], df_rrf['score'].values, width, label='RRF', alpha=0.8)
    ax.bar([i + width/2 for i in x], df_norm['score'].values, width, label='Normalization', alpha=0.8)
    ax.set_ylabel('Score')
    ax.set_title('Scores por Documento')
    ax.set_xticks(x)
    ax.set_xticklabels(docs)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('rrf_vs_normalization.png', dpi=100, bbox_inches='tight')
    plt.show()
    
    print("✓ Gráfico salvo como 'rrf_vs_normalization.png'")
else:
    print("Insuficientes dados para gráfico")

## Tabela de Decisão: Quando Usar Cada Técnica

In [ ]:
# Tabela de decisão
decision_table = pd.DataFrame([
    {
        'Aspecto': 'Algoritmo',
        'RRF': 'Reciprocal Rank Fusion',
        'Normalization': 'Min-Max + Média Aritmética'
    },
    {
        'Aspecto': 'Fórmula',
        'RRF': '1/(k + rank)',
        'Normalization': '(score - min)/(max - min)'
    },
    {
        'Aspecto': 'Sensibilidade a Outliers',
        'RRF': 'Baixa (rank é ordinal)',
        'Normalization': 'Alta (depende de min/max)'
    },
    {
        'Aspecto': 'Escala de Scores',
        'RRF': '[0, 1] (aprox)',
        'Normalization': '[0, 1] (exato)'
    },
    {
        'Aspecto': 'Estabilidade',
        'RRF': 'Muito estável',
        'Normalization': 'Menos estável com outliers'
    },
    {
        'Aspecto': 'Ponderação de Métodos',
        'RRF': 'Baseada em rank',
        'Normalization': 'Baseada em score normalizado'
    },
    {
        'Aspecto': 'Performance',
        'RRF': 'Mais rápido',
        'Normalization': 'Pode ser mais lento'
    },
    {
        'Aspecto': 'Casos de Uso',
        'RRF': 'Busca geral, informação',
        'Normalization': 'Busca com pesos customizados'
    },
    {
        'Aspecto': 'Recomendação',
        'RRF': '✓ Padrão (usar por default)',
        'Normalization': '⚠ Usar com cuidado'
    }
])

print("\n" + "="*100)
print("TABELA DE DECISÃO: RRF vs NORMALIZATION")
print("="*100)
print(decision_table.to_string(index=False))
print("="*100)

## Recomendações Práticas

In [ ]:
recommendations = """
╔════════════════════════════════════════════════════════════════════════════╗
║                      QUANDO USAR CADA TÉCNICA                              ║
╠════════════════════════════════════════════════════════════════════════════╣
║                                                                            ║
║ USE RRF (Reciprocal Rank Fusion) QUANDO:                                  ║
║ ✓ Não há pré-conhecimento sobre distribuição de scores                   ║
║ ✓ Quer garantir estabilidade e robustez                                   ║
║ ✓ Busca é informacional (sem preferências claras)                         ║
║ ✓ Dados têm distribuições diferentes entre BM25 e Vector search          ║
║ ✓ Performance é crítica                                                   ║
║ ✓ É o caso de 90% das aplicações de busca!                              ║
║                                                                            ║
║ EXEMPLO: Assistente jurídico respondendo perguntas sobre leis             ║
║          → RRF combina relevância semântica + por keywords com estabilidade║
║                                                                            ║
╠════════════════════════════════════════════════════════════════════════════╣
║                                                                            ║
║ USE NORMALIZATION (Min-Max) QUANDO:                                       ║
║ ⚠ Conhece a distribuição de scores e quer controlar pesos               ║
║ ⚠ Usa weights específicos (ex: 70% BM25, 30% Vector)                     ║
║ ⚠ Necessita interpretabilidade dos scores                                 ║
║ ⚠ Scores BM25 e Vector são muito desproporcionais                        ║
║                                                                            ║
║ EXEMPLO: E-commerce com pesos customizados                               ║
║          → Texto: 60%, Preço: 20%, Embeddings semânticos: 20%            ║
║                                                                            ║
╠════════════════════════════════════════════════════════════════════════════╣
║                                                                            ║
║ DICA: Combine com métricas!                                               ║
║ - Use NDCG (Normalized Discounted Cumulative Gain) para avaliar ranking   ║
║ - Teste A/B com usuários reais para validar escolha                       ║
║ - RRF geralmente vence em testes (mais robusto)                           ║
║                                                                            ║
╚════════════════════════════════════════════════════════════════════════════╝
"""

print(recommendations)

## Snippet Copiável

In [ ]:
snippet = '''
# ========== SNIPPET: RRF vs NORMALIZATION ==========

# Pipeline RRF (RECOMENDADO)
rrf_pipeline = {
    "description": "Rank Fusion com RRF",
    "processors": [{
        "rrf": {"weights": [0.5, 0.5]}
    }]
}
client.transport.perform_request(
    'PUT', '/_search/pipeline/rrf_pipeline',
    body=rrf_pipeline
)

# Pipeline com Normalization
norm_pipeline = {
    "description": "Normalization com pesos customizados",
    "processors": [{
        "normalization-processor": {
            "normalization": "min_max",
            "combination": {
                "method": "arithmetic_mean",
                "weights": [0.6, 0.4]  # 60% BM25, 40% Vector
            }
        }
    }]
}

# Query híbrida com qualquer pipeline
search_body = {
    "query": {
        "bool": {
            "should": [
                {"multi_match": {"query": "seu_termo", "fields": ["texto"]}},
                {"knn": {"embedding": {"vector": query_embedding, "k": 5}}}
            ]
        }
    },
    "ext": {"search_pipeline": "rrf_pipeline"}  # Trocar por pipeline desejado
}

results = client.search(index="seu_indice", body=search_body)
'''

print(snippet)

## Resumo Executivo

In [ ]:
summary = """
╔════════════════════════════════════════════════════════════════════════════╗
║                       RESUMO: RRF vs NORMALIZATION                          ║
╠════════════════════════════════════════════════════════════════════════════╣
║                                                                            ║
║ MÉTRICA               │ RRF                │ NORMALIZATION                   ║
║ ─────────────────────┼────────────────────┼─────────────────────────────    ║
║ Recomendação          │ ⭐⭐⭐⭐⭐           │ ⭐⭐⭐                         ║
║ Complexidade          │ Simples            │ Média                           ║
║ Estabilidade          │ Muito alta         │ Média                           ║
║ Performance           │ Excelente          │ Boa                             ║
║ Interpretabilidade    │ Fácil              │ Média                           ║
║ Customização          │ Baixa              │ Alta                            ║
║ Ideal para            │ Maioria dos casos  │ Casos com pesos específicos     ║
║                                                                            ║
╠════════════════════════════════════════════════════════════════════════════╣
║                                                                            ║
║ RECOMENDAÇÃO FINAL:                                                       ║
║ Comece com RRF (padrão + robusto).                                        ║
║ Migre para Normalization se tiver razão específica.                       ║
║                                                                            ║
╚════════════════════════════════════════════════════════════════════════════╝
"""

print(summary)